# Week 8 Lab - Data Loading and Initial Analysis

This notebook contains data loading and initial exploration for today's class.

**Expected Data Format:**
- Single CSV file (similar to parquet structure)
- Time series data expected
- Target and weight columns for supervised learning

**Analysis Pipeline:**
1. Data loading and imports
2. Basic data structure analysis
3. Interactive data exploration widgets
4. Time series visualization tools

In [1]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# Display configuration - show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Set up paths (adjust as needed)
base_dir = Path("..")
data_path = base_dir / "data" / "your_data_file.csv"  # Update with actual filename

print("✅ Imports and configuration complete")

✅ Imports and configuration complete


## STEP 1: DATA LOADING

In [2]:
# ============================================
# LOAD DATASET
# ============================================
print("="*60)
print("LOADING DATASET")
print("="*60)

# Load CSV dataset (uncomment and adjust path when ready)
# try:
#     df = pd.read_csv(data_path)
#     print(f"✅ Data loaded successfully!")
#     print(f"Shape: {df.shape}")
# except FileNotFoundError:
#     print("❌ Data file not found. Please update the path.")
#     # Create sample data for demonstration
#     print("Creating sample data for demonstration...")
#     df = create_sample_data()  # You can uncomment this if you have sample data function

# For now, let's create a placeholder structure
print("📝 Ready to load data. Update the path above when your CSV file is ready.")
print("\nExpected structure:")
print("- Time series columns (ts_index, horizon)")
print("- Categorical columns (code, sub_code, sub_category)")
print("- Feature columns (feature_a, feature_b, etc.)")
print("- Target column (y_target)")
print("- Weight column (weight)")

LOADING DATASET
📝 Ready to load data. Update the path above when your CSV file is ready.

Expected structure:
- Time series columns (ts_index, horizon)
- Categorical columns (code, sub_code, sub_category)
- Feature columns (feature_a, feature_b, etc.)
- Target column (y_target)
- Weight column (weight)


## STEP 2: BASIC DATA STRUCTURE ANALYSIS

In [3]:
# ============================================
# BASIC DATA STRUCTURE ANALYSIS
# ============================================
# Uncomment this section after loading your data

print("="*60)
print("DATA STRUCTURE ANALYSIS")
print("="*60)

# Dataset structure (uncomment when data is loaded)
# print("\n📋 DATASET STRUCTURE:")
# print(f"Shape: {df.shape}")
# print(f"Columns: {df.columns.tolist()}")
# print(f"\nData types:")
# print(df.dtypes.value_counts())

# Missing values analysis
# print("\n🔍 MISSING VALUES:")
# missing_data = df.isnull().sum()
# missing_pct = (missing_data / len(df)) * 100
# missing_summary = pd.DataFrame({
#     'Missing Count': missing_data,
#     'Missing %': missing_pct
# })
# print(missing_summary[missing_summary['Missing Count'] > 0])

# Basic statistics
# print("\n📊 BASIC STATISTICS:")
# print(df.describe().T)

print("📝 Uncomment the code above after loading your data to see the analysis.")

DATA STRUCTURE ANALYSIS
📝 Uncomment the code above after loading your data to see the analysis.


## STEP 3: INTERACTIVE DATA EXPLORATION WIDGETS

In [4]:
# ============================================
# INTERACTIVE COLUMN EXPLORER
# ============================================
# This widget will help you explore your data columns

def create_column_explorer(df):
    """Create interactive widgets for column exploration"""
    
    # Get numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    # Create dropdown for column selection
    column_dropdown = widgets.Dropdown(
        options=numeric_cols + categorical_cols,
        description='Select Column:',
        style={'description_width': 'initial'}
    )
    
    # Output area
    output_area = widgets.Output()
    
    def update_column_info(change):
        with output_area:
            clear_output(wait=True)
            col = change['new']
            
            print(f"\n📊 Column Analysis: {col}")
            print("="*40)
            
            # Basic info
            print(f"Data type: {df[col].dtype}")
            print(f"Non-null values: {df[col].count()}")
            print(f"Missing values: {df[col].isnull().sum()}")
            
            if df[col].dtype in ['object', 'category']:
                # Categorical column analysis
                print(f"\n🏷️ Categorical Analysis:")
                print(f"Unique values: {df[col].nunique()}")
                print(f"\nTop 10 values:")
                print(df[col].value_counts().head(10))
            else:
                # Numeric column analysis
                print(f"\n📈 Numeric Analysis:")
                print(f"Min: {df[col].min():.6f}")
                print(f"Max: {df[col].max():.6f}")
                print(f"Mean: {df[col].mean():.6f}")
                print(f"Std: {df[col].std():.6f}")
                
                # Distribution plot
                plt.figure(figsize=(10, 4))
                
                plt.subplot(1, 2, 1)
                sns.histplot(df[col].dropna(), bins=50, kde=True)
                plt.title(f'Distribution of {col}')
                plt.xlabel(col)
                plt.ylabel('Frequency')
                
                plt.subplot(1, 2, 2)
                sns.boxplot(y=df[col].dropna())
                plt.title(f'Box Plot of {col}')
                plt.ylabel(col)
                
                plt.tight_layout()
                plt.show()
    
    # Register callback
    column_dropdown.observe(update_column_info, names='value')
    
    # Display widgets
    display(column_dropdown, output_area)
    
    # Trigger initial display
    if numeric_cols:
        update_column_info({'new': numeric_cols[0]})

print("📝 Column explorer function ready. Uncomment and call after loading data:")
print("create_column_explorer(df)")

📝 Column explorer function ready. Uncomment and call after loading data:
create_column_explorer(df)


## STEP 4: TIME SERIES EXPLORATION WIDGETS

In [5]:
# ============================================
# INTERACTIVE TIME SERIES EXPLORER
# ============================================
# This widget will help you explore time series patterns

def create_time_series_explorer(df):
    """Create interactive widgets for time series exploration"""
    
    # Check for time series columns
    time_cols = [col for col in df.columns if 'ts' in col.lower() or 'time' in col.lower() or 'date' in col.lower()]
    entity_cols = [col for col in df.columns if 'code' in col.lower()]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if not time_cols:
        print("⚠️ No time series columns detected. Looking for index columns...")
        time_cols = [col for col in df.columns if 'index' in col.lower()]
    
    if not entity_cols:
        print("⚠️ No entity code columns detected.")
    
    # Create dropdowns
    feature_dropdown = widgets.Dropdown(
        options=numeric_cols,
        description='Feature:',
        value=numeric_cols[0] if numeric_cols else None
    )
    
    entity_dropdown = widgets.Dropdown(
        options=entity_cols if entity_cols else ['None'],
        description='Entity:',
        value=entity_cols[0] if entity_cols else 'None'
    )
    
    time_dropdown = widgets.Dropdown(
        options=time_cols if time_cols else ['None'],
        description='Time Column:',
        value=time_cols[0] if time_cols else 'None'
    )
    
    # Output area
    output_area = widgets.Output()
    
    def update_time_series_plot(change):
        with output_area:
            clear_output(wait=True)
            
            feature = feature_dropdown.value
            entity_col = entity_dropdown.value
            time_col = time_dropdown.value
            
            if entity_col == 'None' or time_col == 'None':
                print("⚠️ Please select valid entity and time columns.")
                return
            
            # Get unique entities (limit to first 20 for performance)
            unique_entities = df[entity_col].unique()[:20]
            
            # Create entity selector
            entity_selector = widgets.Dropdown(
                options=unique_entities,
                description='Select Entity:',
                value=unique_entities[0]
            )
            
            def plot_entity_series(entity_change):
                with output_area:
                    clear_output(wait=True)
                    
                    entity = entity_change['new']
                    
                    # Filter data for specific entity
                    subset = df[df[entity_col] == entity].sort_values(time_col)
                    
                    if len(subset) == 0:
                        print(f"⚠️ No data found for entity: {entity}")
                        return
                    
                    plt.figure(figsize=(15, 8))
                    
                    # Time series plot
                    plt.subplot(2, 1, 1)
                    plt.plot(subset[time_col], subset[feature], 
                            label=f'{feature}', color='blue', alpha=0.7)
                    plt.plot(subset[time_col], subset[feature].rolling(window=10).mean(),
                            label='Rolling Mean (10)', color='red', linestyle='--')
                    plt.title(f'Time Series: {feature} for {entity_col}: {entity}')
                    plt.xlabel(time_col)
                    plt.ylabel(feature)
                    plt.legend()
                    plt.grid(True, alpha=0.3)
                    
                    # Distribution plot
                    plt.subplot(2, 1, 2)
                    sns.histplot(subset[feature], bins=30, kde=True)
                    plt.title(f'Distribution of {feature} for {entity}')
                    plt.xlabel(feature)
                    plt.ylabel('Frequency')
                    
                    plt.tight_layout()
                    plt.show()
                    
                    # Basic statistics
                    print(f"\n📊 Statistics for {entity_col}: {entity}")
                    print(f"Data points: {len(subset)}")
                    print(f"Time range: {subset[time_col].min()} to {subset[time_col].max()}")
                    print(f"{feature} - Mean: {subset[feature].mean():.4f}, Std: {subset[feature].std():.4f}")
            
            # Register entity selector callback
            entity_selector.observe(plot_entity_series, names='value')
            
            # Display entity selector and trigger initial plot
            display(entity_selector)
            plot_entity_series({'new': unique_entities[0]})
    
    # Register callbacks
    feature_dropdown.observe(update_time_series_plot, names='value')
    entity_dropdown.observe(update_time_series_plot, names='value')
    time_dropdown.observe(update_time_series_plot, names='value')
    
    # Display widgets
    display(widgets.VBox([feature_dropdown, entity_dropdown, time_dropdown]), output_area)

print("📝 Time series explorer function ready. Uncomment and call after loading data:")
print("create_time_series_explorer(df)")

📝 Time series explorer function ready. Uncomment and call after loading data:
create_time_series_explorer(df)


## STEP 5: QUICK DATA SUMMARY

In [6]:
# ============================================
# QUICK DATA SUMMARY FUNCTION
# ============================================

def quick_data_summary(df):
    """Generate a comprehensive data summary"""
    
    print("="*60)
print("QUICK DATA SUMMARY")
    print("="*60)
    
    # Basic info
    print(f"\n📋 Dataset Shape: {df.shape}")
    print(f"📋 Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Column types
    print(f"\n📊 Column Types:")
    print(df.dtypes.value_counts())
    
    # Missing values
    missing_data = df.isnull().sum()
    if missing_data.sum() > 0:
        print(f"\n🔍 Missing Values:")
        missing_summary = missing_data[missing_data > 0]
        for col, count in missing_summary.items():
            pct = (count / len(df)) * 100
            print(f"  {col}: {count} ({pct:.2f}%)")
    else:
        print(f"\n✅ No missing values found!")
    
    # Numeric columns summary
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(f"\n📈 Numeric Columns Summary:")
        print(df[numeric_cols].describe().T[['count', 'mean', 'std', 'min', 'max']])
    
    # Categorical columns summary
    categorical_cols = df.select_dtypes(include=['object']).columns
    if len(categorical_cols) > 0:
        print(f"\n🏷️ Categorical Columns Summary:")
        for col in categorical_cols:
            unique_count = df[col].nunique()
            print(f"  {col}: {unique_count} unique values")
            if unique_count <= 10:
                print(f"    Values: {df[col].unique().tolist()}")

print("📝 Quick summary function ready. Uncomment and call after loading data:")
print("quick_data_summary(df)")

📝 Quick summary function ready. Uncomment and call after loading data:
quick_data_summary(df)


## USAGE INSTRUCTIONS

### When your data is ready:

1. **Update the data path** in the loading section:
   ```python
   data_path = base_dir / "data" / "your_actual_file.csv"
   ```

2. **Load your data** by uncommenting the loading code:
   ```python
   df = pd.read_csv(data_path)
   ```

3. **Run the analysis** by uncommenting the relevant sections:
   ```python
   quick_data_summary(df)
   create_column_explorer(df)
   create_time_series_explorer(df)
   ```

### Features included:
- **Column Explorer**: Interactive dropdown to explore any column
- **Time Series Explorer**: Interactive time series visualization
- **Quick Summary**: Comprehensive data overview
- **Missing Values Analysis**: Detailed missing data report

### Expected for time series data:
- `ts_index` or similar time column
- `code` or similar entity identifier
- Multiple feature columns
- Target and weight columns for supervised learning